In [21]:
# imports
import os
import pandas as pd
import pyodbc
from dotenv import load_dotenv

# google sheets 
import gspread
from gspread_dataframe import set_with_dataframe
from google.oauth2.service_account import Credentials
load_dotenv()

True

In [3]:
# connect to sql server
sql_server_host = os.getenv("sql_server_host")
sql_server_user = os.getenv("sql_server_user")
sql_server_password = os.getenv("sql_server_password")
sql_server_database = os.getenv("sql_server_database")

connection_string = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={sql_server_host};DATABASE={sql_server_database};UID={sql_server_user};PWD={sql_server_password}'

conn = pyodbc.connect(connection_string)

In [4]:
# load data from the server

# orders
orders_query = f"SELECT * FROM Orders"
ordersdf = pd.read_sql(orders_query, conn)

ordersdf

In [ ]:
# invalid_rows = ordersdf[~ordersdf['price_total'].apply(pd.to_numeric, errors='coerce').notnull()]
# invalid_rows 

In [25]:
ordersdf['createdAt'] = pd.to_datetime(ordersdf['createdAt'])
ordersdf = ordersdf[ordersdf['price_total'].apply(pd.to_numeric, errors='coerce').notnull()]
ordersdf['price_total'] = ordersdf['price_total'].astype(float)

rfm_df = ordersdf.groupby('_user').agg(
    recency=('createdAt', 'max'),
    frequency=('createdAt', 'count'),
    monetary_sum=('price_total', 'sum'),
    monetary_avg=('price_total', 'mean')
).reset_index()

rfm_df['monetary_sum'] = rfm_df['monetary_sum'].round(2)
rfm_df['monetary_avg'] = rfm_df['monetary_avg'].round(2)

rfm_df = rfm_df.sort_values(by='frequency', ascending=False)

from datetime import datetime

today = datetime.today()
rfm_df['days_since_last_order'] = (today - rfm_df['recency']).dt.days

rfm_df


,_user,recency,frequency,monetary_sum,monetary_avg,days_since_last_order
18,6245a6f00db8496ee0636dec,2024-12-17 11:12:37.442,1211,631190.80,521.21,26
60,630b4e0295e48e2c08a9e287,2025-01-08 20:11:40.506,446,487741.85,1093.59,3
56,62f14c968a41a318486e146a,2024-12-06 19:07:45.520,331,108609.46,328.13,36
4,61e836019c9def2928a69cc0,2024-12-17 21:58:12.833,310,112971.55,364.42,25
69,638de14ea956cd18f4a95bf5,2025-01-11 17:51:25.265,287,82806.30,288.52,0
...,...,...,...,...,...,...
6579,65ea33b0a7b4701bdc9b39cf,2024-04-05 11:50:31.681,1,628.00,628.00,282
6576,65e9a82dc58e733a048f6a2b,2024-03-17 18:20:12.616,1,2612.00,2612.00,300
6575,65e9a64ac58e733a048f5d1e,2024-03-07 11:36:43.087,1,460.00,460.00,311
6572,65e984c585c293292ccdaef6,2024-03-07 09:15:07.010,1,129.90,129.90,311


In [ ]:
# # upload dataframe to google sheets


# google_sheet_id = os.getenv("GOOGLE_SHEET_ID")
# service_account = os.getenv("SERVICE_ACCOUNT")
# scopes = ["https://www.googleapis.com/auth/spreadsheets"]
# credentials = Credentials.from_service_account_file(service_account, scopes=scopes)

# gc = gspread.authorize(credentials)
# spreadsheet = gc.open_by_key(google_sheet_id)
# worksheet = spreadsheet.sheet1

# set_with_dataframe(worksheet, rfm_df)